In [ ]:
import os, re
from dotenv import load_dotenv
load_dotenv()

from llama_index.core import Settings, VectorStoreIndex, SimpleDirectoryReader, StorageContext
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.node_parser import MarkdownNodeParser

# 1) Models
Settings.llm = OpenAI(model="gpt-4o-mini", temperature=0, api_key=os.getenv("OPENAI_API_KEY"))
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small", api_key=os.getenv("OPENAI_API_KEY"))




METADATA

In [ ]:
def extract_metadata_from_filename(file_path: str) -> dict:
    file_name = os.path.basename(file_path)
    metadata = {"file_name": file_name, "law_year": "unknown", "law_number": "unknown", "category": "قانون"}
    year_match = re.search(r"لسنة\s*(\d{4})", file_name)
    if year_match: metadata["law_year"] = int(year_match.group(1))
    number_match = re.search(r"رقم\s*(\d+)", file_name)
    if number_match: metadata["law_number"] = int(number_match.group(1))
    return metadata

In [ ]:
index_loaded = load_index_from_storage(storage_context)
query_engine = index_loaded.as_chat_engine(llm=Settings.llm, similarity_top_k=5)

2026-03-02 12:19:28,047 - INFO - Loading all indices.


In [ ]:
from pinecone import Pinecone
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
pinecone_index = pc.Index("qanoon")

from llama_index.vector_stores.pinecone import PineconeVectorStore
vector_store = PineconeVectorStore(pinecone_index=pinecone_index)

LOAD DOCS

In [ ]:
reader = SimpleDirectoryReader(
    input_dir="./data",
    required_exts=[".md"],
    recursive=True,
    file_metadata=extract_metadata_from_filename,
)
documents = reader.load_data()


In [ ]:
node_parser = MarkdownNodeParser()


pipeline = IngestionPipeline(
    transformations=[
        node_parser,            
        Settings.embed_model,   
    ],
    vector_store=vector_store  
)

In [ ]:
resp = query_engine.chat("ما هو مضمون قانون رقم 1 لسنة 2024؟")
print(resp)